# 5.22 — Training a Linear Regression Model

This notebook demonstrates a **regression** workflow (predicting a continuous value) using **Linear Regression**.

Key ideas:

- Use a regression target (here: `yield_kg`)
- Split before fitting preprocessing (avoid leakage)
- Compare against a simple baseline (`DummyRegressor`)

CLI equivalent:

```bash
python -m src.main train-regression
```

In [ ]:
import pandas as pd

from sklearn.dummy import DummyRegressor

from src.config import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    EXCLUDED_COLUMNS,
    NUMERICAL_FEATURES,
    TARGET_SOURCE_COLUMN,
)
from src.data_loader import load_training_frame
from src.evaluate_regression import evaluate_regression_model
from src.feature_engineering import engineer_features
from src.preprocessing import (
    fit_preprocessor,
    separate_features_and_target,
    split_data,
    transform_features,
)
from src.regression_model import train_regression_model

## 1) Load data and define a regression target

We use the raw `yield_kg` column directly as a continuous regression target.

In [ ]:
df = load_training_frame()

print("Columns:", list(df.columns))
print("\nTarget summary (yield_kg):")
print(df[TARGET_SOURCE_COLUMN].describe())

df.head()

## 2) Split and preprocess (no leakage)

Rule:

- split into train/test
- **fit** preprocessing on the training set only
- transform train and test using the fitted preprocessor

In [ ]:
X, y = separate_features_and_target(
    df,
    target_column=TARGET_SOURCE_COLUMN,
    feature_columns=[c for c in ALL_FEATURES if c in df.columns],
    excluded_columns=EXCLUDED_COLUMNS,
)

X = engineer_features(X)

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

preprocessor = fit_preprocessor(
    X_train,
    numeric_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    numeric_scaler="standard",
)

X_train_prepared = transform_features(X_train, preprocessor)
X_test_prepared = transform_features(X_test, preprocessor)

X_train_prepared.head()

## 3) Baseline vs Linear Regression

A baseline for regression is often a constant predictor:

- `DummyRegressor(strategy="mean")`: always predicts the training mean.

We evaluate both models using:

- MSE / RMSE
- MAE
- $R^2$

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_prepared, y_train)

linreg = train_regression_model(X_train_prepared, y_train)

baseline_metrics = evaluate_regression_model(baseline, X_test_prepared, y_test)
linreg_metrics = evaluate_regression_model(linreg, X_test_prepared, y_test)

comparison = pd.DataFrame(
    [baseline_metrics, linreg_metrics],
    index=["DummyRegressor(mean)", "LinearRegression"],
)
comparison

## 4) Inspect coefficients

Linear Regression learns a coefficient per prepared feature.

- Positive coefficient: increases prediction as feature increases (all else equal)
- Negative coefficient: decreases prediction as feature increases

In [ ]:
coef = pd.Series(linreg.coef_, index=X_train_prepared.columns)

print("Intercept:", float(linreg.intercept_))

coef.reindex(coef.abs().sort_values(ascending=False).index).head(15)